# Hold the flux DC offsets in a QM that stays open

Every node opens its own QM and closes it at the end, and closing a QM puts its ports back to 0 V. This notebook keeps one QM open on **your** FEMs so the flux lines stay at the chosen voltage between nodes.

What the cells do:
1. Connect, choose the qubits (cell 2), check the plan.
2. **Activate**: open the held QM (controller config only, offsets as idle values), write the same offsets to the QuAM state (`port.offset`), create `~/.qualibrate/qm_hold.json`.
   While that file exists, `qmm.open_qm()` (used by `qm_session`) returns the held QM instead of opening a new one, and each job is sent with the node's elements / pulses / waveforms (the port settings stay those of the held QM; if a node changes port settings in memory, they are sent too and the differences are logged). No file in site-packages is touched.
3. Run your nodes as usual.
4. **Last cell, after all experiments**: restore the state, close the held QM, delete the flag file. Everything is back to the normal behaviour.

Requirements: QOP >= 3.5 and qm-qua >= 1.4 (checked in the activate cell).
While a QM is held, nobody else can use the ports of the listed FEMs. Do not run `00_close_other_qms.py` / `close_all_qms()` in between (it closes the held QM, the voltage goes to 0 V).

In [ ]:
# 1. Connect
from quam_libs.components import QuAM
from quam_libs.lib import qm_hold

machine = QuAM.load()
qmm = machine.connect()
print(qm_hold.status(qmm))

In [ ]:
# 2. Choose what to hold
QUBITS = None   # qubits whose z line is held (z.joint_offset). None = every qubit; machine.active_qubit_names = only the active ones
PAIRS = None            # None = every coupler between two chosen qubits (coupler.decouple_offset), or a list of pair names
OVERRIDES = {}          # optional voltage per qubit / pair name, e.g. {"q4": 0.12}
EXTRA_FEMS = []         # optional extra FEMs to own, e.g. [("con1", 5)]

plan = qm_hold.plan_hold(machine, QUBITS, PAIRS, OVERRIDES, EXTRA_FEMS)
qm_hold.print_plan(plan)

In [ ]:
# 3. ACTIVATE. Check the plan above first (voltages and FEMs).
flag = qm_hold.activate(machine, qmm, plan)
print(qm_hold.status(qmm))

In [ ]:
# 4. Optional check: the idle offsets of the held QM
held = qmm.get_qm(flag["qm_id"])
for con, con_cfg in held.get_config()["controllers"].items():
    for fem, fem_cfg in con_cfg.get("fems", {}).items():
        for port, port_cfg in fem_cfg.get("analog_outputs", {}).items():
            if port_cfg.get("offset"):
                print(con, fem, port, port_cfg["offset"])

---
# Appendix: update the held QM to the current state.json (without closing it)

Use this when the QM is held and you changed the state (`z.joint_offset`, `coupler.decouple_offset`, a delay, a filter, ...) and want the held QM to follow, **without** deactivating and activating again (the ports would go to 0 V in between).

The cell below reads the current `state.json`, then calls `qm.update_config` on the held QM with:
- the offset of every held flux line = the current `joint_offset` (qubit z line) / `decouple_offset` (coupler), also written to the port in the state;
- every other port setting (delay, filter, ...) = what the state generates now.

Only the FEMs of the held QM are sent. Do it while no node is running. If the server refuses a field, you get an error and nothing is saved (the state and the flag file stay as they were).
Only the `offset` is documented as changeable on an open QM; the other fields are decided by the server (`shareable` and octave parameters are refused).
Run it only when you want to update the held QM.

In [ ]:
# Appendix: make the held QM (offsets, delay, filter, ...) match the current state.json
qm_hold.sync_held_to_state(qmm)
print(qm_hold.status(qmm))

---
## Run your nodes now (from the IDE, `_run_file.py`, or qualibrate).

## When all experiments are done, run the LAST cell below.

In [ ]:
# LAST CELL: deactivate. Restores the state, closes the held QM (ports go to 0 V), removes the flag file.
qm_hold.deactivate(qmm, close_qm=True)
print(qm_hold.status(qmm))